In [ ]:
# ── Pipeline settings ─────────────────────────────────────────────────────────
# Read by every notebook of this pipeline (start_run in NB_Utils_Common and the layer utils).
VL_NAME = 'VL_PROC'

# Table locations: location → VL key of its lakehouse. Each location's schema comes from VL key
# <LOCATION>_SCHEMA (BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA, MD_SILVER_SCHEMA, OPS_SCHEMA) and is
# 'dbo' when that key is absent: add the key to the VL to move a layer to another schema.
LAKEHOUSES = {
    'bronze':    'BRONZE_LH_ABFSS',
    'silver':    'SILVER_LH_ABFSS',
    'gold':      'PROC_GOLD_LH_ABFSS',
    'md_silver': 'MD_SILVER_LH_ABFSS',
    'md_gold':   'MD_GOLD_LH_ABFSS',
    'ops':       'OPS_LH_ABFSS',
    # ▶▶ NEW 22-Oct-2026 · Order_Status_Purchase ◀◀
    # SD Silver, read only: SD owns the delivery items (LIPS), so PROC looks up which deliveries reference a PO item
    # instead of loading LIPS again (NB_Gold_PROC, order status rule 10). A link lookup, as Audit does with 'proc_silver'.
    # Optional: add VL key SD_SILVER_LH_ABFSS (and SD_SILVER_SCHEMA when not dbo) to VL_PROC. Without it the status
    # uses the delivery reference on the EKES confirmation only. Run SD Silver before PROC Gold.
    'sd_silver': 'SD_SILVER_LH_ABFSS',
}

# Bronze source: default VL key and schema; a table's source_lh_key / source_schema override them.
# DAB lakehouses store tables under /Tables/dbo/ (since 23 Nov 2026).
SOURCE_LH_KEY = 'DAB_PROC_LH_ABFSS'
SOURCE_SCHEMA = 'Tables/dbo'   # ▶ 23 Nov 2026: B1 moved to the dbo schema (same lakehouse, tables under Tables/dbo/)

# Optional overrides — the defaults live in the utils:
# BRONZE_MODE          = 'cdc'        # 'cdc' | 'overwrite' (true replace: no history, no soft deletes)
# DESC_BRONZE_MODE     = 'overwrite'  # description tables
# SILVER_MODE          = 'full'       # 'full' | 'incremental' | 'overwrite' (true replace; SKs kept)
# SILVER_LOOKBACK_DAYS = 7            # incremental: Bronze rows changed in the last N days
#                                     #   (never later than the last successful run − SILVER_WINDOW_MARGIN_MINUTES, 60)
# GOLD_MODE            = 'merge'      # 'merge' | 'overwrite' (true replace, unless a write_gold call passes mode)
# GOLD_DELETE_MISSING  = False        # True: Gold MERGE also deletes rows whose source is gone
# GOLD_DELETE_MAX_PCT  = 10           # ...unless that would remove more than this % of a table
# DQ_FAIL_ON_CRITICAL  = False        # True: the end-of-run report raises after logging
# GOLD_PROFILE_STRINGS = True        # end of each Gold run: text column sizes measured, GOLD_STRING_LENGTHS proposed (17 Nov 2026)
# GOLD_PERSIST         = True        # write_gold keeps the incoming rows in memory / on disk for its checks and write
# SILVER_PERSIST       = True        # process_table keeps the working rows in memory / on disk (Bronze read once)
# RESOURCE_PROFILE     = {}          # e.g. {'GOLD': 'readHeavyForPBI'}: Fabric resource profile per layer, set in start_run
# DELTA_AUTO_COMPACT   = False       # True: Delta auto compaction after the session's writes
# DQ_THRESHOLDS        = {'sk_null_pct': 1.0, 'desc_null_pct': 5.0}   # merged over DQ_DEFAULTS
# Per table (in its config entry): 'bronze_mode': 'overwrite', 'silver_mode': 'incremental',
#   'silver_lookback_days': 14, 'gold_mode': 'overwrite', 'dq': {'sk_null_pct': 5.0}.
# Silver notebooks also take run parameters RUN_LOOKBACK_DAYS, RUN_FROM_DATE, RUN_FORCE_FULL.

In [ ]:
# Each table entry contains:
#   source_lh_key — VL key identifying the DAB source lakehouse for this table
#   primary_keys  — columns used to identify a unique row (SAP field names)
#   sk_col        — surrogate key column added in Silver
#   gold_name     — target Fact table name in Gold
#   silver_cols   — SAP field names to keep in Silver
#   rename_map    — SAP field name → friendly business name
#   desc_joins    — description text joins (resolved in Silver from PROC or MD Silver)
#   sk_joins      — surrogate key joins from MD Silver dims (resolved in Silver)
#   id_cols       — computed ID columns
#   lzero_cols    — friendly column names to strip leading zeros from
#   enrich        — post-write enrichment steps run by _apply_enrichments() in NB_Utils_Silver
#                   Supported types: pre_join, derived, sign, xrate, unit_conv, mbew_marc, drop

PROC_TABLE_CONFIGS = {
    'EKKO': {
        'source_lh_key': 'DAB_PROC_LH_ABFSS',
        'primary_keys': ['EBELN'],
        'sk_col':       'PO_SK',
        'gold_name':    'Fact_PO_LineItem',
        'silver_cols':  [
            'EBELN','BUKRS','BSTYP','BSART','LOEKZ','STATU','AEDAT','ERNAM',
            'LIFNR','SPRAS','ZTERM','EKORG','BEDAT','EKGRP','INCO1','INCO2','WAERS','RESWK',
            # EA audit reports (v2 set)
            'FRGGR',
            'FRGSX',
            'FRGKE',
            'FRGZU',
            'FRGRL',
            'WKURS',
        ],
        'rename_map': {
            'EBELN': 'Purchasing_Document',  'BUKRS': 'Company_Code',
            'BSTYP': 'Purch_Doc_Category',   'BSART': 'Purchasing_Doc_Type',
            'LOEKZ': 'Deletion_Indicator',   'STATU': 'Status',
            'AEDAT': 'Created_On',           'ERNAM': 'Created_By',
            'LIFNR': 'Supplier',             'SPRAS': 'Language_Key',
            'ZTERM': 'Payment_Terms',        'EKORG': 'Purch_Organization',
            'BEDAT': 'Document_Date',        'EKGRP': 'Purchasing_Group',
            'INCO1': 'Incoterms',            'INCO2': 'Incoterms_2',
            'WAERS': 'Currency',             'RESWK': 'Supplying_Plant',
            # EA audit reports (v2 set)
            'FRGGR': 'Release_Group',
            'FRGSX': 'Release_Strategy',
            'FRGKE': 'Release_Indicator',
            'FRGZU': 'Release_Status',
            'FRGRL': 'Subject_To_Release',
            'WKURS': 'Exchange_Rate',
        },
        'desc_joins': [
            {'desc_table':'T001T', 'join_key':'Company_Code',        'desc_key':'BUKRS',              'desc_col':'TXTNA',       'alias':'Company_Code_Name'},
            {'desc_table':'T161T', 'join_key':'Purchasing_Doc_Type', 'desc_key':'BSART',              'desc_col':'BATXT',       'alias':'Purchasing_Doc_Type_Desc'},
            {'desc_table':'T024E', 'join_key':'Purch_Organization',  'desc_key':'Purch_Organization', 'desc_col':'Description', 'alias':'Purch_Organization_Name'},
            {'desc_table':'T024',  'join_key':'Purchasing_Group',    'desc_key':'Purchasing_Group',   'desc_col':'Purchasing_Group_Name', 'alias':'Purchasing_Group_Name'},
            # EA audit reports (v2 set)
            {'desc_table': 'T16FT', 'join_keys': ['Release_Group', 'Release_Strategy'], 'desc_keys': ['FRGGR', 'FRGSX'],
             'desc_col': 'FRGXT', 'alias': 'Release_Strategy_Desc'},
            {'desc_table': 'T16FH', 'join_key': 'Release_Group',     'desc_key': 'FRGGR', 'desc_col': 'FRGGT', 'alias': 'Release_Group_Desc'},
            {'desc_table': 'T16FE', 'join_key': 'Release_Indicator', 'desc_key': 'FRGKE', 'desc_col': 'FRGET', 'alias': 'Release_Indicator_Desc'},
        ],
        'sk_joins': [
            {'src_table': 'LFA1',  'join_key': 'Supplier',           'src_key': 'Supplier',           'sk_col': 'Vendor_SK'},
            {'src_table': 'T001',  'join_key': 'Company_Code',       'src_key': 'Company_Code',       'sk_col': 'Company_Code_SK'},
            {'src_table': 'T024E', 'join_key': 'Purch_Organization', 'src_key': 'Purch_Organization', 'sk_col': 'Purch_Org_SK'},
            {'src_table': 'T024',  'join_key': 'Purchasing_Group',   'src_key': 'Purchasing_Group',   'sk_col': 'Purchasing_Group_SK'},
        ],
        'id_cols': [
            {'col': 'Purchasing_Doc_Type_ID', 'sources': ['Purchasing_Doc_Type', 'Purchasing_Doc_Type_Desc']},
            {'col': 'Incoterms_Header_ID',    'sources': ['Incoterms', 'Incoterms_2']},
        ],
        'lzero_cols': ['Purchasing_Document', 'Supplier'],
        'enrich': [
            # Ordering address (EKPA partner 'BA' at header level: no vendor subrange, no plant), lowest
            # partner counter: the partner vendor number and its Dim_Vendor key
            {'type': 'sk_lookup', 'join_key': 'Purchasing_Document', 'src_table': 'EKPA', 'src_key': 'Purchasing_Document',
             'src_filter': "Partner_Function = 'BA' AND (Vendor_Subrange IS NULL OR trim(Vendor_Subrange) = '') "
                           "AND (Plant IS NULL OR trim(Plant) = '')",
             'order_by': ['Partner_Counter'],
             'src_sk_col': 'Partner_Vendor_SK', 'sk_col': 'Order_From_Vendor_SK',
             'src_cols': {'Partner_Vendor': 'Order_From_Vendor'}},
        ],
    },


    'EKPO': {
        'source_lh_key': 'DAB_PROC_LH_ABFSS',
        'primary_keys': ['EBELN','EBELP'],
        'sk_col':       'PO_Item_SK',
        'gold_name':    'Fact_PO_LineItem',
        'silver_cols':  [
            'EBELN','EBELP','LOEKZ','AEDAT','TXZ01','MATNR','BUKRS','WERKS',
            'LGORT','MEINS','PLIFZ','MATKL','PSTYP','INFNR','BPRME','CREATIONDATE',
            'ELIKZ','KO_PRCTR','KNTTP','INCO1','INCO2','KZABS','MENGE','NETPR','NETWR','PEINH',
            # EA audit reports (v2 set)
            'REPOS',
            'WEBRE',
            'EREKZ',
            'WEPOS',
            'BSTAE',
            'STAPO',
            'RETPO',
            'UMREZ',
            'UMREN',
            'BPUMZ',
            'BPUMN',
            'WEBAZ',
            'AFNAM',
            'BANFN',
            'LABNR',
            'LBLKZ',
            # Procurement feedback (SAP table list v14)
            'BEDNR',
            'ABSKZ',   # ▶ NEW (Order_Status): rejection indicator (rule 1, item rejected = X)
            'ADRNR',   # New ( added on Nov 2nd)
            'BNFPO',   # New ( added on Nov 2nd)
            'UNTTO',   # ▶ 6 Nov 2026 (order status, Jeremy rule 2 / C22): under-delivery tolerance %, closes a line within it
            'EGLKZ',   # ▶ 6 Nov 2026 (order status, C24): final delivery indicator, EA early exit 8

        ],
        'rename_map': {
            'EBELN': 'Purchasing_Document',  'EBELP': 'Item',
            'LOEKZ': 'Deletion_Indicator',   'AEDAT': 'Last_Changed_On',
            'TXZ01': 'Short_Text',           'MATNR': 'Material',
            'BUKRS': 'Company_Code',         'WERKS': 'Plant',
            'LGORT': 'Storage_Location',     'MEINS': 'Order_Unit',
            'PLIFZ': 'Planned_Deliv_Time',   'MATKL': 'Material_Group',
            'PSTYP': 'Item_Category',        'INFNR': 'Purchasing_Info_Rec',
            'BPRME': 'Order_Price_Unit',     'CREATIONDATE': 'Creation_Date',
            'ELIKZ': 'Delivery_Completed',   'KO_PRCTR': 'Profit_Center',
            'KNTTP': 'Acct_Assignment_Cat',  'INCO1': 'Incoterms',
            'INCO2': 'Incoterms_2',          'KZABS': 'Acknowledgment_Reqd',
            'MENGE': 'Order_Quantity',       'NETPR': 'Net_Order_Price',
            'NETWR': 'Net_Order_Value',      'PEINH': 'Price_Unit',
            # EA audit reports (v2 set)
            'REPOS': 'Invoice_Receipt_Ind',
            'WEBRE': 'GR_Based_Inv_Verif',
            'EREKZ': 'Final_Invoice',
            'WEPOS': 'Goods_Receipt_Ind',
            'BSTAE': 'Confirmation_Control',
            'STAPO': 'Statistical_Item',
            'RETPO': 'Returns_Item',
            'UMREZ': 'Conv_To_Base_Num',
            'UMREN': 'Conv_To_Base_Den',
            'BPUMZ': 'Qty_Conversion_Num',
            'BPUMN': 'Qty_Conversion_Den',
            'WEBAZ': 'GR_Processing_Days',
            'AFNAM': 'Requisitioner',
            'BANFN': 'Purchase_Requisition',
            'LABNR': 'Order_Acknowledgment',
            'LBLKZ': 'Subcontracting_Vendor',
            'BEDNR': 'Req_Tracking_Number',
            'ABSKZ': 'Rejection_Indicator',   # ▶ NEW (Order_Status)
            'ADRNR': 'Address',   # ▶ NEW (added on Nov 2nd)
            'BNFPO': 'Item_Of_Requisition', # ▶ NEW (added on Nov 2nd)
            'UNTTO': 'Underdelivery_Tolerance',   # ▶ 6 Nov 2026 (order status)
            'EGLKZ': 'Final_Delivery',            # ▶ 6 Nov 2026 (order status)
        },
        'desc_joins': [
            {'desc_table':'T001T', 'join_key':'Company_Code',    'desc_key':'BUKRS', 'desc_col':'TXTNA',     'alias':'Company_Code_Name'},
            {'desc_table':'T001W', 'join_key':'Plant',           'desc_key':'Plant', 'desc_col':'Plant_Name','alias':'Plant_Name'},
            {'desc_table':'T001L', 'join_keys':['Plant','Storage_Location'],'desc_keys':['WERKS','LGORT'],'desc_col':'LGOBE','alias':'Storage_Location_Desc'},
            # EA audit reports (v2 set)
            {'desc_table': 'T163Y', 'join_key': 'Item_Category', 'desc_key': 'PSTYP',
             'desc_cols': [{'col': 'EPSTP', 'alias': 'Item_Category_Ext'}, {'col': 'PTEXT', 'alias': 'Item_Category_Desc'}]},
        ],
        'sk_joins': [
            {'src_table': 'MARA',  'join_key': 'Material',             'src_key': 'Material',     'sk_col': 'Material_SK'},
            {'src_table': 'T001W', 'join_key': 'Plant',                'src_key': 'Plant',        'sk_col': 'Plant_SK'},
            {'src_table': 'EINA',  'join_key': 'Purchasing_Info_Rec',  'src_key': 'Info_Record',  'sk_col': 'PIR_SK'},
            # ▶ 6 Nov 2026: links to EBAN (Fact_Purch_Req) and EKPV (Fact_PO_Stock_Transfer_Shipping). Two-column keys go in
            #   join_keys / src_keys (plural); a list in join_key makes the lookup fail and leaves the key empty on every row.
            {'src_table': 'EBAN',  'join_keys': ['Purchase_Requisition', 'Item_Of_Requisition'],
             'src_keys':  ['Purchase_Requisition', 'Item_Of_Requisition'], 'sk_col': 'Purchase_Requisition_SK'},
            {'src_table': 'EKPV',  'join_keys': ['Purchasing_Document', 'Item'],
             'src_keys':  ['Purchasing_Document', 'Purchasing_Document_Item'], 'sk_col': 'PO_Stock_SK'},
        ],
        'id_cols': [
            {'col': 'Incoterms_Item_ID', 'sources': ['Incoterms', 'Incoterms_2']},
        ],
        'lzero_cols': ['Purchasing_Document', 'Item', 'Profit_Center', 'Purchase_Requisition', 'Item_Of_Requisition'],
        'enrich': [
            # Pull Currency, Created_On, and Purch_Organization from EKKO header.
            # Purch_Organization is needed for the compound EINE join below; dropped after.
            {'type': 'pre_join', 'from_table': 'EKKO',
             'join_key': 'Purchasing_Document',
             'cols': ['Currency', 'Created_On', 'Purch_Organization'],
             'drop_after': ['Currency', 'Created_On', 'Purch_Organization']},
            # Resolve PIR_Purch_Org_SK from EINE (MD Silver) — compound join on Info_Record+Purch_Org.
            # EINE is the combined Dim_PIR grain; its PK is PIR_Purch_Org_SK (INFNR+EKORG).
            # src_keys maps EINE's column names to EKPO's (Info_Record → Purchasing_Info_Rec).
            {'type': 'pre_join', 'from_table': 'EINE',
             'use_md_silver': True,
             'join_key': ['Purchasing_Info_Rec', 'Purch_Organization'],
             'src_keys': ['Info_Record', 'Purch_Organization'],
             'cols': ['PIR_Purch_Org_SK'],
             'drop_after': []},
            {'type': 'derived', 'col': 'Price_Per_Unit',
             'numerator': 'Net_Order_Price', 'denominator': 'Price_Unit'},
            {'type': 'xrate',
             'currency_col': 'Currency', 'date_col': 'Created_On',
             'amount_cols': ['Net_Order_Price', 'Net_Order_Value', 'Price_Per_Unit']},
            {'type': 'mbew_marc'},
            # team edit (21 Oct 2026): material per plant (MARC), the key of Dim_Material_Plant
            {'type': 'sk_lookup', 'join_keys': ['Material', 'Plant'], 'src_table': 'MARC',
             'src_keys': ['Material', 'Plant'], 'sk_col': 'Material_Plant_SK'},
            # ▶ 18 Nov 2026 (business approved the base-unit rule, EA reports 11 / 14): factor order unit → base unit.
            #   Base_Factor_PO = EKPO UMREZ / UMREN; Base_Factor_MARM = MARM numerator / denominator of the order unit, the
            #   fallback. Gold multiplies quantities by COALESCE(Base_Factor_PO, Base_Factor_MARM); no factor → empty, never 1.
            #   The MARM key is resolved with sk_lookup (trims and strips leading zeros on both sides), then read by that key.
            {'type': 'sk_lookup', 'join_keys': ['Material', 'Order_Unit'], 'src_table': 'MARM',
             'src_keys': ['Material', 'Alternative_Unit_Of_Measure'], 'sk_col': 'Material_UoM_SK'},
            {'type': 'pre_join', 'from_table': 'MARM', 'use_md_silver': True,
             'join_key': 'Material_UoM_SK', 'cols': ['Numerator', 'Denominator'],
             'drop_after': ['Numerator', 'Denominator']},
            {'type': 'derived', 'col': 'Base_Factor_PO',
             'numerator': 'Conv_To_Base_Num', 'denominator': 'Conv_To_Base_Den'},
            {'type': 'derived', 'col': 'Base_Factor_MARM',
             'numerator': 'Numerator', 'denominator': 'Denominator'},
        ],
    },


    'EKET': {
        'source_lh_key': 'DAB_PROC_LH_ABFSS',
        'primary_keys': ['EBELN','EBELP','ETENR'],
        'sk_col':       'PO_Schedule_SK',
        'gold_name':    'Fact_PO_Schedule',
        'silver_cols':  [
            'EBELN','EBELP','ETENR','SLFDT','EINDT','WEMNG','WAMNG',
            'BEDAT','MENGE','AMENG','LPEIN','BANFN','BNFPO','WADAT',
            # EA audit reports (v2 set)
            'DAT01',
            'MNG02',
            # Procurement feedback (SAP table list v14): quantity delivered for stock transfer orders
            'GLMNG',
            'DABMG',   # ▶ NEW (Order_Status): quantity reduced for MRP (schedule residual = MENGE - WEMNG - DABMG)
        ],
        'rename_map': {
            'EBELN': 'Purchasing_Document',  'EBELP': 'Item',
            'ETENR': 'Schedule_Line',        'SLFDT': 'Stat_Rel_Del_Date',
            'EINDT': 'Delivery_Date',        'WEMNG': 'Quantity_Delivered',
            'WAMNG': 'Issued_Quantity',      'BEDAT': 'Purchase_Order_Date',   # order date of the schedule line (v14 name)
            'MENGE': 'Scheduled_Quantity',   'AMENG': 'Previous_Quantity',
            'LPEIN': 'Deliv_Date_Category',  'BANFN': 'Purchase_Requisition',
            'BNFPO': 'Item_Of_Requisition',  'WADAT': 'Goods_Issue_Date',
            # EA audit reports (v2 set)
            'DAT01': 'Committed_Date',
            'MNG02': 'Committed_Quantity',
            'GLMNG': 'Qty_Delivered_STO',
            'DABMG': 'Qty_Reduced_MRP',   # ▶ NEW (Order_Status)
        },
        'desc_joins': [],
        'sk_joins': [
            # purchase order header and item keys (Dim_PO_Document / the PO facts)
            {'src_table': 'EKKO', 'join_key': 'Purchasing_Document', 'src_key': 'Purchasing_Document', 'sk_col': 'PO_SK'},
            {'src_table': 'EKPO', 'join_keys': ['Purchasing_Document', 'Item'],
             'src_keys': ['Purchasing_Document', 'Item'], 'sk_col': 'PO_Item_SK'},
        ],
        'id_cols':    [],
        'lzero_cols': ['Purchasing_Document', 'Item', 'Schedule_Line',
                       'Purchase_Requisition', 'Item_Of_Requisition'],
        'enrich': [
            # Pull Order_Unit + Material from EKPO Silver — both needed for MARM-driven unit_conv.
            # drop_after defers until all enrich steps complete (same pattern as EKBE).
            {'type': 'pre_join', 'from_table': 'EKPO',
             'join_key': ['Purchasing_Document', 'Item'],
             'cols': ['Order_Unit', 'Material'],
             'drop_after': ['Order_Unit', 'Material']},
            # Bilateral KG/LB conversion via MARM — produces Scheduled_Quantity_KG and _LB.
            # All materials, not only consumables (Procurement, SAP table list v14: "conversion for all GPL"): product_rule
            # False. A material with no KG / LB entry in MARM gets an empty _KG / _LB quantity (weight unknown).
            {'type': 'unit_conv', 'qty_col': 'Scheduled_Quantity', 'uom_col': 'Order_Unit',
             'primary_unit': 'KG', 'conv_unit': 'LB', 'product_rule': False},
            {'type': 'unit_conv', 'qty_col': 'Scheduled_Quantity', 'uom_col': 'Order_Unit',
             'primary_unit': 'LB', 'conv_unit': 'KG', 'product_rule': False},
        ],
    },

    'EKBE': {
        'source_lh_key': 'DAB_PROC_LH_ABFSS',
        'primary_keys': ['EBELN','EBELP','ZEKKN','VGABE','GJAHR','BELNR','BUZEI'],
        'sk_col':       'PO_History_SK',
        'gold_name':    'Fact_PO_Receipts',
        'silver_cols':  [
            'EBELN','EBELP','ZEKKN','VGABE','GJAHR','BELNR','BUZEI',
            'MENGE','DMBTR','BUDAT','BLDAT','SHKZG','BEWTP','BWART',
            'WAERS','ELIKZ','MATNR','WERKS','ERNAM','CHARG',
            'ETENS',   # ▶ NEW (Order_Status): confirmation sequence of the history record (exact match to an EKES segment)
        ],
        'rename_map': {
            'EBELN': 'Purchasing_Document',  'EBELP': 'Item',
            'ZEKKN': 'Seq_No_Of_Account',    'VGABE': 'Transevent_Type',
            'GJAHR': 'Material_Doc_Year',    'BELNR': 'Material_Document',
            'BUZEI': 'Material_DocItem',     'MENGE': 'Quantity',
            'DMBTR': 'Amount_In_LC',         'BUDAT': 'Posting_Date',
            'BLDAT': 'Document_Date',        'SHKZG': 'DebitCredit_Ind',
            'BEWTP': 'PO_History_Category',  'BWART': 'Movement_Type',
            'WAERS': 'Currency',             'ELIKZ': 'Delivery_Completed',
            'MATNR': 'Material',             'WERKS': 'Plant',
            'ERNAM': 'Created_By',           'CHARG': 'Batch',
            'ETENS': 'Confirmation_Seq',     # ▶ NEW (Order_Status)
        },
        'desc_joins': [
            {'desc_table':'T163C',  'join_key':'PO_History_Category','desc_key':'BEWTP','desc_col':'BEWTL','alias':'PO_History_Category_Desc'},
            {'desc_table':'T156HT', 'join_key':'Movement_Type',      'desc_key':'BWART','desc_col':'BTEXT','alias':'Movement_Type_Desc'},
            {'desc_table':'T001W',  'join_key':'Plant',              'desc_key':'Plant','desc_col':'Plant_Name','alias':'Plant_Name'},
        ],
        'sk_joins': [
            {'src_table': 'MARA',  'join_key': 'Material', 'src_key': 'Material', 'sk_col': 'Material_SK'},
            {'src_table': 'T001W', 'join_key': 'Plant',    'src_key': 'Plant',    'sk_col': 'Plant_SK'},
            # purchase order header and item keys (Dim_PO_Document / the PO facts)
            {'src_table': 'EKKO', 'join_key': 'Purchasing_Document', 'src_key': 'Purchasing_Document', 'sk_col': 'PO_SK'},
            {'src_table': 'EKPO', 'join_keys': ['Purchasing_Document', 'Item'],
             'src_keys': ['Purchasing_Document', 'Item'], 'sk_col': 'PO_Item_SK'},
        ],
        'id_cols': [
            {'col': 'PO_History_Category_ID', 'sources': ['PO_History_Category', 'PO_History_Category_Desc']},
            {'col': 'Movement_Type_ID',       'sources': ['Movement_Type',       'Movement_Type_Desc']},
        ],
        'lzero_cols': ['Purchasing_Document', 'Item'],
        'enrich': [
            {'type': 'sign',
             'indicator_col': 'DebitCredit_Ind', 'negative_val': 'H',
             'cols': [('Quantity', 'Signed_Quantity'),
                      ('Amount_In_LC', 'Signed_Amount_LC')]},
            {'type': 'xrate',
             'currency_col': 'Currency', 'date_col': 'Posting_Date',
             'amount_cols': ['Signed_Amount_LC'],
             'rename': {
                 'Signed_Amount_LC_USD_EOM': 'Signed_Amount_USD_EOM',
                 'Signed_Amount_LC_USD_FIX': 'Signed_Amount_USD_FIX',
                 'Signed_Amount_LC_EUR_EOM': 'Signed_Amount_EUR_EOM',
                 'Signed_Amount_LC_EUR_FIX': 'Signed_Amount_EUR_FIX',
             }},
            {'type': 'pre_join', 'from_table': 'EKPO',
             'join_key': ['Purchasing_Document', 'Item'],
             'cols': ['Order_Unit'],
             'drop_after': ['Order_Unit']},
            {'type': 'unit_conv', 'qty_col': 'Quantity', 'uom_col': 'Order_Unit',
             'primary_unit': 'KG', 'conv_unit': 'LB'},
            {'type': 'unit_conv', 'qty_col': 'Quantity', 'uom_col': 'Order_Unit',
             'primary_unit': 'LB', 'conv_unit': 'KG'},
            {'type': 'sign',
             'indicator_col': 'DebitCredit_Ind', 'negative_val': 'H',
             'cols': [('Quantity_KG', 'Signed_Quantity_KG'),
                      ('Quantity_LB', 'Signed_Quantity_LB')]},
            {'type': 'drop', 'cols': ['Quantity_KG', 'Quantity_LB']},
        ],
    },
}

print(f'Config loaded: {list(PROC_TABLE_CONFIGS.keys())}')

In [ ]:
# ── EA audit reports (v2 set): purchasing tables (keys + needed columns only) ─────────────────

# EKES — vendor confirmations / inbound deliveries per PO item (confirmed order due date)
PROC_TABLE_CONFIGS['EKES'] = {
    'primary_keys': ['EBELN', 'EBELP', 'ETENS'],
    'sk_col':       'PO_Confirmation_SK',
    'gold_name':    'Fact_PO_Confirmation',
    'silver_cols':  ['EBELN', 'EBELP', 'ETENS', 'EBTYP', 'EINDT', 'ERDAT',
                     # ▶ NEW (Order_Status): confirmation quantity, MRP reduction, delivery reference, deletion flag
                     'MENGE', 'DABMG', 'VBELN', 'VBELP', 'LOEKZ',
                     'KZDIS'],                                  # ▶ Audit phase 2 (1 Nov 2026): MRP-relevant confirmation (D32), read by Order_Due_Date in NB_Gold_PROC
    'rename_map': {
        'EBELN': 'Purchasing_Document',
        'EBELP': 'Item',
        'ETENS': 'Confirmation_Seq',
        'EBTYP': 'Confirmation_Category',
        'EINDT': 'Confirmed_Delivery_Date',
        'ERDAT': 'Created_On',
        'MENGE': 'Confirmation_Quantity',          # ▶ NEW (Order_Status)
        'KZDIS': 'MRP_Relevant',               # ▶ Audit phase 2 (1 Nov 2026): 'X' = MRP relevant, kept as is
        'DABMG': 'Confirmation_Qty_Reduced_MRP',   # ▶ NEW (Order_Status)
        'VBELN': 'Inbound_Delivery',               # ▶ NEW (Order_Status)
        'VBELP': 'Inbound_Delivery_Item',          # ▶ NEW (Order_Status)
        'LOEKZ': 'Confirmation_Deletion',          # ▶ NEW (Order_Status)
    },
    'desc_joins': [],
    'id_cols':    [],
    'sk_joins': [
        # purchase order header and item keys (Dim_PO_Document / the PO facts)
        {'src_table': 'EKKO', 'join_key': 'Purchasing_Document', 'src_key': 'Purchasing_Document', 'sk_col': 'PO_SK'},
        {'src_table': 'EKPO', 'join_keys': ['Purchasing_Document', 'Item'],
         'src_keys': ['Purchasing_Document', 'Item'], 'sk_col': 'PO_Item_SK'},
    ],
    'lzero_cols': ['Purchasing_Document', 'Item'],
}


# EKKN — PO account assignment (order, G/L account)
PROC_TABLE_CONFIGS['EKKN'] = {
    'primary_keys': ['EBELN', 'EBELP', 'ZEKKN'],
    'sk_col':       'PO_Acct_Assignment_SK',
    'gold_name':    'Fact_PO_LineItem',       # Order, GL_Account of the first assignment per item
    'silver_cols':  ['EBELN', 'EBELP', 'ZEKKN', 'AUFNR', 'SAKTO', 'LOEKZ'],
    'rename_map': {
        'EBELN': 'Purchasing_Document',
        'EBELP': 'Item',
        'ZEKKN': 'Acct_Assignment_No',
        'AUFNR': 'Order',
        'SAKTO': 'GL_Account',
        'LOEKZ': 'Deletion_Indicator',
    },
    'desc_joins': [],
    'id_cols':    [],
    'sk_joins':   [],
    'lzero_cols': ['Purchasing_Document', 'Item', 'GL_Account'],
}


# EKPA — PO partner functions, header level (Order from = PARVW 'BA')
PROC_TABLE_CONFIGS['EKPA'] = {
    'primary_keys': ['EBELN', 'EKORG', 'LTSNR', 'WERKS', 'PARVW', 'PARZA'],
    'sk_col':       'PO_Partner_SK',
    'gold_name':    'Fact_PO_LineItem',       # Order_From_Vendor (PARVW 'BA', header level)
    'silver_cols':  ['EBELN', 'EKORG', 'LTSNR', 'WERKS', 'PARVW', 'PARZA', 'LIFN2'],
    'rename_map': {
        'EBELN': 'Purchasing_Document',
        'EKORG': 'Purch_Organization',
        'LTSNR': 'Vendor_Subrange',
        'WERKS': 'Plant',
        'PARVW': 'Partner_Function',
        'PARZA': 'Partner_Counter',
        'LIFN2': 'Partner_Vendor',
    },
    'desc_joins': [],
    'id_cols':    [],
    'sk_joins': [
        # Dim_Vendor key of the partner vendor (EKKO uses it for the ordering address)
        {'src_table': 'LFA1', 'join_key': 'Partner_Vendor', 'src_key': 'Supplier',
         'src_sk_col': 'Vendor_SK', 'sk_col': 'Partner_Vendor_SK'},
    ],
    'lzero_cols': ['Purchasing_Document', 'Partner_Vendor'],
}


# EKAB — release documentation for contracts (release order date)
PROC_TABLE_CONFIGS['EKAB'] = {
    'primary_keys': ['KONNR', 'KTPNR', 'EBELN', 'EBELP'],
    'sk_col':       'Contract_Release_SK',
    'gold_name':    'Fact_PO_Contract_Release',
    'silver_cols':  ['KONNR', 'KTPNR', 'EBELN', 'EBELP', 'BEDAT'],
    'rename_map': {
        'KONNR': 'Contract',
        'KTPNR': 'Contract_Item',
        'EBELN': 'Purchasing_Document',
        'EBELP': 'Item',
        'BEDAT': 'Release_Order_Date',
    },
    'desc_joins': [],
    'id_cols':    [],
    'sk_joins': [
        # PO keys of the release order
        {'src_table': 'EKKO', 'join_key': 'Purchasing_Document', 'src_key': 'Purchasing_Document', 'sk_col': 'PO_SK'},
        {'src_table': 'EKPO', 'join_keys': ['Purchasing_Document', 'Item'],
         'src_keys': ['Purchasing_Document', 'Item'], 'sk_col': 'PO_Item_SK'},
    ],
    'lzero_cols': ['Contract', 'Contract_Item', 'Purchasing_Document', 'Item'],
}

print('EA audit report tables added: EKES, EKKN, EKPA, EKAB')


In [ ]:
# EKPV 
PROC_TABLE_CONFIGS['EKPV'] = {
    'primary_keys': ['EBELN', 'EBELP'],
    'sk_col':       'PO_Stock_SK',
    'gold_name':    'Fact_PO_Stock_Transfer_Shipping',
    'silver_cols':  ['EBELN', 'EBELP', 'VSTEL', 'LPRIO', 'ROUTE', 'KUNNR', 'KUNAG', 'SPDNR', 'VKORG', 'VTWEG', 'SPART', 'LEDAT', 'PLIFZ', 'SPRAS', 'VSBED', 'LADGR', 'TRAGR', 'ABLAD', 'ZZKZAZU'],
    'rename_map': {
        'EBELN': 'Purchasing_Document',
        'EBELP': 'Purchasing_Document_Item',
        'VSTEL': 'Shipping_Point',
        'LPRIO': 'Delivery_Priority',
        'ROUTE': 'Route',
        'KUNNR': 'Customer',
        'KUNAG': 'Sold_To_Party',
        'SPDNR': 'Forwarding_Agent',
        'VKORG': 'Sales_Organization',
        'VTWEG': 'Distribution_Channel',
        'SPART': 'Division',
        'LEDAT': 'Delivery_Creation_Time',
        'PLIFZ': 'Planned_Delivery_Time',
        'SPRAS': 'Language',
        'VSBED': 'Shipping_Conditions',
        'LADGR': 'Loading_Group',
        'TRAGR': 'Transportation_Group',
        'ABLAD': 'Unloading_Point',
        'ZZKZAZU': 'Order_Combination'
    },
    'desc_joins': [],
    'id_cols':    [],
    'sk_joins': [],
    'lzero_cols': ['Purchasing_Document', 'Purchasing_Document_Item', 'Customer', 'Sold_To_Party'],
}

# EBKN 
PROC_TABLE_CONFIGS['EBKN'] = {
    'primary_keys': ['BANFN', 'BNFPO', 'ZEBKN'],
    'sk_col':       'Purchase_Requisition_Account_SK',
    'gold_name':    'Fact_Purch_Req_Account_Assignment',
    'silver_cols':  ['BANFN', 'BNFPO', 'ZEBKN', 'LOEKZ', 'ERDAT', 'ERNAM', 'MENGE', 'VPROZ', 'SAKTO', 'GSBER', 'KOSTL', 'VBELN', 'POSNR', 'VETEN', 'AUFNR', 'WEMPF', 'ABLAD', 'KOKRS', 'PRCTR'],   # 6 Nov 2026: KOKRS (was KOKRAS)
    'rename_map': {
        'BANFN': 'Purchase_Requisition',
        'BNFPO': 'Item_Of_Requisition',
        'ZEBKN': 'Serial_Number_Account_Assignment',
        'LOEKZ': 'Deletion_Indicator',
        'ERDAT': 'Created_On',
        'ERNAM': 'Created_By',
        'MENGE': 'Quantity_Requested',
        'VPROZ': 'Distribution',
        'SAKTO': 'GL_Account',
        'GSBER': 'Business_Area',
        'KOSTL': 'Cost_Center',
        'VBELN': 'Sales_Document',
        'POSNR': 'Sales_Document_Item',
        'VETEN': 'Schedule_Line',
        'AUFNR': 'Order',
        'WEMPF': 'Goods_Recipient',
        'ABLAD': 'Unloading_Point',
        'KOKRS': 'Controlling_Area',
        'PRCTR': 'Profit_Center',
    },
    'desc_joins': [],
    'id_cols':    [],
    'sk_joins': [],
    'lzero_cols': ['Purchase_Requisition', 'Item_Of_Requisition', 'GL_Account', 'Cost_Center', 'Sales_Document', 'Sales_Document_Item', 'Schedule_Line', 'Order'],
    'enrich': [   # 6 Nov 2026: uom_col is the unit (MEINS); it read Cost_Center
        # EBKN has no unit field: its quantity is in the requisition item's unit, EBAN MEINS. Read it from EBAN Silver first.
        {'type': 'pre_join', 'from_table': 'EBAN', 'join_key': ['Purchase_Requisition', 'Item_Of_Requisition'],
         'cols': ['Unit_Of_Measure']},
        {'type': 'unit_conv', 'qty_col': 'Quantity_Requested', 'uom_col': 'Unit_Of_Measure',
         'primary_unit': 'KG', 'conv_unit': 'LB'},
        {'type': 'unit_conv', 'qty_col': 'Quantity_Requested', 'uom_col': 'Unit_Of_Measure',
         'primary_unit': 'LB', 'conv_unit': 'KG'},
    ],
}

# EBAN 
PROC_TABLE_CONFIGS['EBAN'] = {
    'primary_keys': ['BANFN', 'BNFPO'],
    'sk_col':       'Purchase_Requisition_SK',
    'gold_name':    'Fact_Purch_Req',
    'silver_cols':  ['BANFN', 'BNFPO', 'BSART', 'BSTYP', 'BSAKZ', 'LOEKZ', 'STATU', 'ESTKZ', 'FRGKZ', 'FRGZU', 'FRGST', 'EKGRP', 'ERNAM', 'ERDAT', 'AFNAM', 'MATNR', 'TXZ01', 'WERKS', 'LGORT', 'MATKL', 'RESWK', 'MENGE', 'MEINS', 'BUMNG', 'BADAT', 'LFDAT', 'FRGDT', 'PREIS', 'PEINH', 'PSTYP', 'KNTTP', 'WEPOS', 'FLIEF', 'EKORG', 'DISPO', 'EBELN', 'EBELP', 'BEDAT', 'BSMNG', 'CHARG', 'WAERS', 'PLIFZ', 'LIFNR', 'LBLKZ',
                    'EBAKZ'],   # ▶ 3 Nov 2026: closed flag, read by SC (Fact_Purchase_Requisition)
    'rename_map': {
        'BANFN': 'Purchase_Requisition',
        'BNFPO': 'Item_Of_Requisition',
        'BSART': 'Document_Type',
        'BSTYP': 'Document_Category',
        'BSAKZ': 'Control_Indicator',
        'LOEKZ': 'Deletion_Indicator',   # 6 Nov 2026: was 'Deletion-Indicator'
        'STATU': 'Processing_Status',
        'ESTKZ': 'Creation_Indicator',
        'FRGKZ': 'Release_Indicator',
        'FRGZU': 'Release_Status',
        'FRGST': 'Release_Strategy',
        'EKGRP': 'Purchasing_Group',
        'ERNAM': 'Created_By',
        'ERDAT': 'Changed_On',
        'AFNAM': 'Requisitioner',
        'MATNR': 'Material',
        'TXZ01': 'Short_Text',
        'WERKS': 'Plant',
        'LGORT': 'Storage_Location',
        'MATKL': 'Material_Group',
        'RESWK': 'Supplying_Plant',
        'MENGE': 'Requisition_Quantity',
        'MEINS': 'Unit_Of_Measure',
        'BUMNG': 'Shortage_Quantity',
        'BADAT': 'Requisition_Date',
        'LFDAT': 'Delivery_Date',
        'FRGDT': 'Release_Date',
        'PREIS': 'Valuation_Price',
        'PEINH': 'Price_Unit',
        'PSTYP': 'Item_Category',
        'KNTTP': 'Acct_Assignment_Category',
        'WEPOS': 'Goods_Receipt',
        'FLIEF': 'Fixed_Vendor',
        'EKORG': 'Purchasing_Organization',
        'DISPO': 'MRP_Controller',
        'EBELN': 'Purchase_Document',
        'EBELP': 'Purchase_Document_Item',
        'BEDAT': 'Purchase_Document_Date',
        'BSMNG': 'Quantity_Ordered',
        'CHARG': 'Batch',
        'WAERS': 'Currency',
        'PLIFZ': 'Planned_Delivery_Time',
        'LIFNR': 'Desired_Vendor',
        'LBLKZ': 'Subcontracting_Vendor',
        'EBAKZ': 'Requisition_Closed',   # ▶ 3 Nov 2026, for SC
    },
    'desc_joins': [],
    'id_cols':    [],
    'sk_joins': [
        {'src_table': 'MARA', 'join_key': 'Material', 'src_key': 'Material', 'sk_col': 'Material_SK'},
        {'src_table': 'MARC', 'join_keys': ['Material', 'Plant'], 'src_keys': ['Material', 'Plant'],
         'sk_col': 'Material_Plant_SK'},   # 6 Nov 2026: Material + Plant (Material alone picked any plant)
        {'src_table': 'T001W', 'join_key': 'Plant', 'src_key': 'Plant', 'sk_col': 'Plant_SK'},   # ▶ 3 Nov 2026, for SC
        # 6 Nov 2026: removed the join of EBAN to itself (PO_Delivery_Status_SK). EBAN's own Purchase_Requisition_SK is the
        # link; EKPO carries it (EKPO sk_joins), so Fact_PO_Delivery_Status relates to Fact_Purch_Req through it.
    ],
    'lzero_cols': ['Purchase_Requisition', 'Item_Of_Requisition', 'Fixed_Vendor', 'Purchase_Document', 'Purchase_Document_Item', 'Desired_Vendor', 'Subcontracting_Vendor'],
    'enrich': [   # 6 Nov 2026: uom_col is the unit (MEINS); it read Price_Unit (PEINH, a number)
        {'type': 'unit_conv', 'qty_col': 'Requisition_Quantity', 'uom_col': 'Unit_Of_Measure',
         'primary_unit': 'KG', 'conv_unit': 'LB'},
        {'type': 'unit_conv', 'qty_col': 'Requisition_Quantity', 'uom_col': 'Unit_Of_Measure',
         'primary_unit': 'LB', 'conv_unit': 'KG'},
    ],
}

print('New tables added:EKPV, EBKN, EBAN')

In [ ]:
# NOT LIVE (gated, see the text block above): every line stays commented. Needs business sign-off D10 and HSWAE in the EKBE extract (D11).
# # ── Known issue K1: EKBE currency for the USD and EUR values (config change; changes existing columns) ──
# # DECISIONS (made, reasons in brackets):
# #   D10 Apply the fix. [The current USD and EUR values are wrong for foreign-currency POs: the local amount DMBTR
# #       is converted with the PO currency. The change is needed to report USD and EUR correctly.]
# #       Only the business sign-off on the changed USD and EUR figures is outstanding; this block stays commented until then.
# #   D11 Add HSWAE (company-code currency) at source. [The config cannot use a column that the Bronze extract does not
# #       load. Check it exists in the EKBE source table first; if not, the DAB team adds it.]
# #   D12 Until D10 is signed off, report local-currency values only. [Local values are correct today.]
# # Check after the rebuild: for a foreign-currency PO, Signed_Amount_USD_FIX = Signed_Amount_LC x rate on the posting date.
# PROC_TABLE_CONFIGS['EKBE']['silver_cols'] += ['HSWAE']                                   # D11: company-code currency
# PROC_TABLE_CONFIGS['EKBE']['rename_map']['HSWAE'] = 'Local_Currency'
# _xr = [e for e in PROC_TABLE_CONFIGS['EKBE']['enrich'] if e.get('type') == 'xrate']       # the step that builds the USD / EUR values
# _xr[0]['currency_col'] = 'Local_Currency'                                                  # D10: DMBTR is in company-code currency, not PO currency

In [ ]:
# ▶▶ NEW 20-Oct-2026 · Order_Status_Purchase · settings (read by NB_Gold_PROC and NB_Check_Order_Status_PROC) ◀◀
# Sources: EA Support case 05670302 (14 Oct 2026) and "Open PO v2 - Engineering Logic and Record Allocation" (14 Oct 2026).
OS_TOLERANCE = 0.001                       # quantity tolerance for every threshold (Jeremy, section 6)

# EKES confirmation categories (EBTYP) behind each status. ▶ CHANGED 6 Nov 2026 (Jeremy, C1 / C25): Lincoln's customizing
# has only AB (order acknowledgment) and ZS (LE shipping notification with receipt); GW is also in the data. RS, SN, CF,
# GA, FC, LC, ZB and LA do not exist, so rules 7, 12 and 13 have no category (0 rows) until Jeremy says to drop them.
# was: RECEIPT ['RS'], SHIPPING ['SN', 'GW'], CALL ['CF'], AVAILABLE ['GA'], CONFIRMED ['FC', 'LC', 'AB', 'ZB', 'ZS', 'LA']
OS_CAT_RECEIPT   = []                                        # rule 7  Receipt from supplier
OS_CAT_SHIPPING  = ['ZS', 'GW']                              # rule 11 Shipping notification
OS_CAT_CALL      = []                                        # rule 12 Call from supplier
OS_CAT_AVAILABLE = []                                        # rule 13 Goods available at supplier
OS_CAT_CONFIRMED = ['AB']                                    # rule 15 Confirmed

# The rules of Order_Status_Purchase, checked in this order, the first one that is true wins. Rule 1 reads
# "Cancelled partially" when goods or invoices were received. The condition of each rule is its flag (NB_Gold_PROC).
# ▶ CHANGED 6 Nov 2026 (Jeremy): rule 0 Return item (EKPO RETPO, C20: flagged, then excluded from the reports) is checked
# first; rule 6 Valuated GR Blocked Stock is retired (movement 107 is an advance shipping notice: it now counts as
# Shipping notification, rule 11). The numbers of the other rules are unchanged.
OS_RULES = [
    (0,  'Return item'),
    (1,  'Cancelled completely'),
    (2,  'Closed by GR'),
    (3,  'Closed by order creation or IR'),
    (4,  'Blocked'),
    (5,  'GR blocked stock (103)'),
    (7,  'Receipt from supplier'),
    (8,  'Service entry sheet created'),
    (9,  'Goods issued'),
    (10, 'Outbound delivery created'),
    (11, 'Shipping notification'),
    (12, 'Call from supplier'),
    (13, 'Goods available at supplier'),
    (14, 'Stock committed'),
    (15, 'Confirmed'),
    (16, 'To be ordered'),
    (17, 'To be confirmed'),
    (18, 'Ordered'),
]
# The flag of rules 0 to 17, in rule order; they are stored on Fact_PO_Order_Status_Evidence (since 15 Nov 2026; the
# fact Fact_PO_LineItem_Schedule keeps only Order_Status_Rule and Order_Status_Purchase).
OS_RULE_FLAGS = ['Is_Return_Item', 'Is_Cancelled', 'Is_Closed_By_GR', 'Is_Closed_By_IR', 'Is_Blocked', 'Is_GR_Blocked',
                 'Is_Receipt_From_Supplier', 'Is_Service_Entry', 'Is_Goods_Issued', 'Is_Outbound_Delivery',
                 'Is_Shipping_Notification', 'Is_Call_From_Supplier', 'Is_Goods_Available', 'Is_Stock_Committed',
                 'Is_Confirmed', 'Is_To_Be_Ordered', 'Is_To_Be_Confirmed']


In [ ]:
# NOT LIVE (gated, see the text block above): every line stays commented. Needs business sign-off D10 and HSWAE in the EKBE extract (D11).
# # ── Known issue K1: EKBE currency for the USD and EUR values (config change; changes existing columns) ──
# # DECISIONS (made, reasons in brackets):
# #   D10 Apply the fix. [The current USD and EUR values are wrong for foreign-currency POs: the local amount DMBTR
# #       is converted with the PO currency. The change is needed to report USD and EUR correctly.]
# #       Only the business sign-off on the changed USD and EUR figures is outstanding; this block stays commented until then.
# #   D11 Add HSWAE (company-code currency) at source. [The config cannot use a column that the Bronze extract does not
# #       load. Check it exists in the EKBE source table first; if not, the DAB team adds it.]
# #   D12 Until D10 is signed off, report local-currency values only. [Local values are correct today.]
# # Check after the rebuild: for a foreign-currency PO, Signed_Amount_USD_FIX = Signed_Amount_LC x rate on the posting date.
# PROC_TABLE_CONFIGS['EKBE']['silver_cols'] += ['HSWAE']                                   # D11: company-code currency
# PROC_TABLE_CONFIGS['EKBE']['rename_map']['HSWAE'] = 'Local_Currency'
# _xr = [e for e in PROC_TABLE_CONFIGS['EKBE']['enrich'] if e.get('type') == 'xrate']       # the step that builds the USD / EUR values
# _xr[0]['currency_col'] = 'Local_Currency'                                                  # D10: DMBTR is in company-code currency, not PO currency